# Phase 3B — ProcessPoolExecutor vs ThreadPool and BLAS

**Pilot first.** This notebook uses the real 20K × 384 MiniLM embedding matrix. Compare same top-10 search workloads under sequential NumPy, batched NumPy, ThreadPool, and ProcessPool. The ProcessPool is created **outside the timed region** and initialized with the corpus once per pool; time and record initialization separately. Also measure the cost of serializing the corpus as a diagnostic. Results are not available until execution.

Processes bypass the Python GIL but incur startup, inter-process communication and memory duplication; NumPy BLAS already releases the GIL. A process pool may lose even when its computations run concurrently. CPU-only; 2 BLAS/Numba threads not assumed to mean 2 physical cores. This is not a Phase 1 pure-Python baseline.

In [ ]:
%pip -q install threadpoolctl pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,time,pickle,platform,multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor,ThreadPoolExecutor
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits,threadpool_info
ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
EMB=ROOT/'phase2_matrix_ops/2A_sparse_dense/qwen_20k_minilm_384_float32.npy'
OUT=ROOT/'phase3_parallelism/benchmarks';FIG=ROOT/'phase3_parallelism/figures'
OUT.mkdir(parents=True,exist_ok=True);FIG.mkdir(parents=True,exist_ok=True)
X=np.load(EMB).astype(np.float32,copy=False)
assert X.shape==(20000,384),X.shape
X=np.ascontiguousarray(X/np.maximum(np.linalg.norm(X,axis=1,keepdims=True),1e-12))
print('CPU:',os.cpu_count(),'BLAS:',threadpool_info())

In [ ]:
# Top-level functions are required for ProcessPool pickling.
_PROCESS_CORPUS=None
def process_init(corpus):
    global _PROCESS_CORPUS
    _PROCESS_CORPUS=corpus
    # Avoid nested BLAS oversubscription inside every process.
    threadpool_limits(limits=1,user_api='blas')
def topk(scores,k=10):
    ids=np.arange(scores.shape[0])
    return np.lexsort((ids,-scores))[:k].astype(np.int64)
def search_one(q,corpus,k=10):
    return topk(corpus@q,k)
def process_search(q):
    return search_one(q,_PROCESS_CORPUS)
def search_sequential(Q,corpus):
    return [search_one(q,corpus) for q in Q]
def search_batched(Q,corpus):
    scores=Q@corpus.T
    return [topk(row) for row in scores]
def search_threadpool(Q,corpus,workers):
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return list(ex.map(lambda q:search_one(q,corpus),Q))
def search_processpool(pool,Q,chunksize):
    return list(pool.map(process_search,Q,chunksize=chunksize))

# Colab/Linux uses fork; process pool functions must be defined in a code cell,
# not a nested function. If fork is unavailable, run as a standalone Python module.
print('Available start methods:',mp.get_all_start_methods())

In [ ]:
PILOT=True  # Change to False only after reviewing pilot outputs.
N_DOCS=[1000,5000] if PILOT else [1000,5000,10000,20000]
N_QUERIES=[8,32] if PILOT else [16,64,128]
WORKERS=[1,2] if PILOT else [1,2,4]
CHUNKSIZES=[1,8] if PILOT else [1,8,32]
REPEATS=3 if PILOT else 5
BLAS_THREADS=1
SEED=20261020
assert 'fork' in mp.get_all_start_methods(),'This notebook currently requires Linux fork start method.'
ctx=mp.get_context('fork')
records=[];init_records=[]
with threadpool_limits(limits=BLAS_THREADS,user_api='blas'):
 for n in N_DOCS:
  corpus=X[:n]
  for nq in N_QUERIES:
   rng=np.random.default_rng(SEED+n+nq)
   ids=rng.choice(n,size=nq,replace=nq>n)
   Q=np.ascontiguousarray(corpus[ids])
   ref=search_sequential(Q,corpus)
   configs=[('sequential',0,0,lambda:search_sequential(Q,corpus)),
            ('batched_blas',0,0,lambda:search_batched(Q,corpus))]
   configs += [('threadpool',w,1,lambda w=w:search_threadpool(Q,corpus,w)) for w in WORKERS]
   for name,w,ch,fn in configs:
    got=fn()
    if name=='batched_blas':
     assert all(set(a)==set(b) for a,b in zip(ref,got)),(name,n,nq)
    else:
     assert all(np.array_equal(a,b) for a,b in zip(ref,got)),(name,n,nq,w)
    for rep in range(REPEATS):
     t0=time.perf_counter_ns();fn();ms=(time.perf_counter_ns()-t0)/1e6
     records.append(dict(mode='pilot' if PILOT else 'full',method=name,workers=w,chunksize=ch,n_docs=n,n_queries=nq,blas_threads=BLAS_THREADS,repeat=rep,elapsed_ms=ms))
   for w in WORKERS:
    t_init=time.perf_counter_ns()
    with ProcessPoolExecutor(max_workers=w,mp_context=ctx,initializer=process_init,initargs=(corpus,)) as pool:
     # Force worker startup before timing (lazy fork), and verify all results.
     got=search_processpool(pool,Q,chunksize=1)
     startup_ms=(time.perf_counter_ns()-t_init)/1e6
     init_records.append(dict(n_docs=n,n_queries=nq,workers=w,startup_and_first_search_ms=startup_ms))
     assert all(np.array_equal(a,b) for a,b in zip(ref,got)),('processpool',n,nq,w)
     for ch in CHUNKSIZES:
      for rep in range(REPEATS):
       t0=time.perf_counter_ns();got=search_processpool(pool,Q,ch);ms=(time.perf_counter_ns()-t0)/1e6
       records.append(dict(mode='pilot' if PILOT else 'full',method='processpool',workers=w,chunksize=ch,n_docs=n,n_queries=nq,blas_threads=BLAS_THREADS,repeat=rep,elapsed_ms=ms))
   print('done',n,nq)
tag='pilot' if PILOT else 'full'
df=pd.DataFrame(records);startup=pd.DataFrame(init_records)
df.to_csv(OUT/f'phase3b_processpool_raw_{tag}.csv',index=False)
startup.to_csv(OUT/f'phase3b_processpool_startup_{tag}.csv',index=False)
print('Timed rows',len(df),'startup rows',len(startup))

In [ ]:
keys=['method','workers','chunksize','n_docs','n_queries','blas_threads']
s=df.groupby(keys,as_index=False).agg(median_ms=('elapsed_ms','median'),mean_ms=('elapsed_ms','mean'),std_ms=('elapsed_ms','std'))
b=s[s.method=='sequential'][['n_docs','n_queries','median_ms']].rename(columns={'median_ms':'baseline_ms'})
s=s.merge(b,on=['n_docs','n_queries'],how='left')
s['speedup_vs_sequential']=s.baseline_ms/s.median_ms
s.to_csv(OUT/f'phase3b_processpool_summary_{tag}.csv',index=False)
print(s.to_string(index=False))
print('Process startup and first-search:',startup.to_string(index=False))

In [ ]:
focus=s[(s.n_docs==max(N_DOCS))&(s.n_queries==max(N_QUERIES))].copy()
focus['label']=focus.apply(lambda r: r.method if r.method in ('sequential','batched_blas') else f'{r.method} w={int(r.workers)} c={int(r.chunksize)}',axis=1)
fig,ax=plt.subplots(figsize=(12,5))
ax.bar(focus.label,focus.speedup_vs_sequential)
ax.axhline(1,color='gray',linestyle='--')
ax.set_title(f'Phase 3B: {max(N_DOCS)} docs × {max(N_QUERIES)} queries; BLAS=1')
ax.set_ylabel('Speedup vs sequential NumPy (median)');ax.tick_params(axis='x',rotation=65)
fig.tight_layout();p=FIG/f'phase3b_processpool_speedup_{tag}.png';fig.savefig(p,dpi=160);plt.show()
print('Figure saved',p)

## Interpretation checklist

- Compare same corpus, query count and top-10; separate process startup cost from steady-state searches.
- The Linux `fork` path inherits memory pages copy-on-write; this is **not** the same as measuring pickling/copying the entire corpus on every task. Each query and result still incurs IPC/serialization overhead.
- GIL: separate processes can run Python bytecode concurrently; NumPy BLAS releases the GIL already, so process-based NumPy search may not win.
- Amdahl: process scheduling, IPC, ranking and limited CPU resources cap speedup. Investigate non-monotonic or unusually high speedups.
- Code complexity: compare process pool setup and lifecycle to batched NumPy, which is simpler.
- Report pilot separately. After sharing pilot results, set `PILOT=False` and rerun only after confirming correctness and timings.